In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql.functions import desc

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, 'silver_output', 'parquet_data_hhs')
GOLD_PARQUET_DIR = os.path.join(BASE_DIR, 'gold_output', 'parquet_data_hhs')
os.makedirs(GOLD_PARQUET_DIR, exist_ok=True)

print(f"SILVER_PARQUET_DIR: {SILVER_PARQUET_DIR}")
print(f"GOLD_PARQUET_DIR:   {GOLD_PARQUET_DIR}")

In [0]:
# Read silver layer parquet tables

df_benefits = spark.read.parquet(
    os.path.join(SILVER_PARQUET_DIR, 'benefits_grouped_transformed')
)
df_rates = spark.read.parquet(
    os.path.join(SILVER_PARQUET_DIR, 'rate_baseline')
)

In [0]:
rates_stats_df = df_rates.agg(
    F.min("IndividualRate").alias("MinIndividualRate"),
    F.max("IndividualRate").alias("MaxIndividualRate"),
)
display(rates_stats_df)

In [0]:
benefits_stats_df = df_benefits.agg(
    F.min("BenefitCount").alias("MinBenefitCount"),
    F.max("BenefitCount").alias("MaxBenefitCount")
)
display(benefits_stats_df)


In [0]:
df_rates = df_rates.filter((F.col("IndividualRate") != 0) & (F.col("IndividualRate") <= 1000))

In [0]:
df_benefits_rates = df_rates.join(df_benefits, on="PlanId", how="inner")
display(df_benefits_rates)

In [0]:
df_benefits_rates.write.mode("overwrite").parquet(
    os.path.join(GOLD_PARQUET_DIR, 'benefits_rates')
)

In [0]:
import pandas as pd

pdf_avg = (
    df_benefits_rates
    .groupBy("BenefitCount")
    .agg(F.avg("IndividualRate").alias("AvgIndividualRate"))
    .toPandas()
)

# Group BenefitCount into ranges of 10
bins = range(0, 220, 50)
labels = [f"{i+1}–{i+50}" for i in range(0, 200, 50)]
pdf_avg["BenefitRange"] = pd.cut(pdf_avg["BenefitCount"], bins=bins, labels=labels)

pdf_grouped = (
    pdf_avg.groupby("BenefitRange", observed=True)["AvgIndividualRate"]
    .mean()
    .reset_index()
)

overall_avg = pdf_grouped["AvgIndividualRate"].mean()

plt.figure(figsize=(14, 6))
plt.bar(pdf_grouped["BenefitRange"], pdf_grouped["AvgIndividualRate"], color="steelblue", alpha=0.8)
plt.axhline(overall_avg, color="tomato", linewidth=1.5, linestyle="--", label=f"Overall avg (${overall_avg:.2f})")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Benefit Count Range")
plt.ylabel("Avg Individual Rate ($)")
plt.title("Average Individual Rate by Benefit Count Range")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

pdf = df_benefits_rates.select("BenefitCount", "IndividualRate").toPandas()

# Linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(pdf["BenefitCount"], pdf["IndividualRate"])
x_line = np.linspace(pdf["BenefitCount"].min(), pdf["BenefitCount"].max(), 200)
y_line = slope * x_line + intercept

plt.figure(figsize=(10, 6))
plt.scatter(pdf["BenefitCount"], pdf["IndividualRate"], alpha=0.4, color="steelblue", edgecolors="none", s=40, label="Plans")
plt.plot(x_line, y_line, color="tomato", linewidth=2, label=f"Trend line (R²={r_value**2:.3f}, slope={slope:.2f})")
plt.xlabel("Benefit Count")
plt.ylabel("Individual Rate ($)")
plt.title("Individual Rate vs Benefit Count (per Plan)")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
from scipy import stats

# Pearson correlation via Spark
pearson_corr = df_benefits_rates.stat.corr("BenefitCount", "IndividualRate")

# Spearman correlation & p-value via scipy (on collected sample)
pdf_corr = df_benefits_rates.select("BenefitCount", "IndividualRate").toPandas()
spearman_corr, p_value = stats.spearmanr(pdf_corr["BenefitCount"], pdf_corr["IndividualRate"])

# R2 measure (coefficient of determination) for linear regression
slope, intercept, r_value, _, _ = stats.linregress(pdf_corr["BenefitCount"], pdf_corr["IndividualRate"])
r2 = r_value ** 2

print(f"Pearson  correlation (linear):  {pearson_corr:.4f}")
print(f"Spearman correlation (monotonic): {spearman_corr:.4f}  (p-value: {p_value:.4e})")
print(f"R² (coefficient of determination): {r2:.4f}")
print()
if abs(spearman_corr) >= 0.7:
    strength = "strong"
elif abs(spearman_corr) >= 0.4:
    strength = "moderate"
else:
    strength = "weak"
direction = "positive" if spearman_corr > 0 else "negative"
print(f"Interpretation: {strength} {direction} monotonic relationship between BenefitCount and IndividualRate.")
print(f"Statistically significant: {'Yes' if p_value < 0.05 else 'No'} (alpha=0.05)")

Pearson = 0.77 — There is a strong linear component to the relationship. As BenefitCount increases, IndividualRate tends to increase in a roughly proportional way, not just in rank order.

Spearman = 0.68 — The monotonic (rank-based) correlation is slightly lower than Pearson. This is expected given the threshold jump visible in the bar chart: the relationship isn't perfectly monotonic for all ranges (rates oscillate at low counts), but the overall direction is consistently positive. The gap between Pearson and Spearman (0.77 vs 0.68) hints that the linear fit is actually somewhat inflated by the extreme values at high BenefitCounts.

R² = 0.59 — BenefitCount alone explains ~59% of the variance in IndividualRate. That's a meaningful predictor, but 41% of the variation comes from other factors — plan type, region, insurer pricing, etc.

Overall conclusion: BenefitCount is a significant and moderately strong driver of IndividualRate (p ≈ 0, so not due to chance). However, the relationship is not smooth — the bar chart shows a clear structural break around count 41–50, suggesting plans above that threshold belong to a fundamentally different tier. A simple linear model would overestimate rates at low counts and underestimate at high ones; segmenting by tier before modeling would likely improve explanatory power.

**Do IndividualRates increase with BenefitCount? Yes — with a clear structural break.**

| Measure | Value | Meaning |
|---|---|---|
| Pearson | 0.77 | Strong linear relationship |
| Spearman | 0.68 | Moderate monotonic relationship |
| R² | 0.59 | BenefitCount explains ~59% of rate variance |
| p-value | ~0.00 | Statistically significant |

**Key finding**: The relationship is not gradual. Plans with `BenefitCount` up to ~40 have low average rates ($10–$55), while plans above ~50 benefits jump sharply to $200–$500. This threshold effect — visible in both the scatter plot trend line and the bar chart — suggests two distinct plan tiers rather than a smooth pricing scale.

**Practical takeaway**: `BenefitCount` is a meaningful predictor of cost, but the 41% unexplained variance means other factors (region, insurer, plan type) also play a role. Any pricing or segmentation model should account for the tier break at ~BenefitCount 49 rather than applying a single linear assumption across all plans.

# Unit Tests